### The Agentic Loop

In agent.ipynb, we did function calling by hand. We sent a message and got back a function call. We ran it, sent the result back, and got the answer.

That works for one function call. It breaks down when the model wants to search several times, or when the first search misses the answer. We don't know in advance how many calls the model will want. So we need a loop that keeps calling the model and running tools until it's done. 

An agent is exactly that.

### Anatomy of an agent
With the LLM in the driver's seat, we have an agent. It's an AI assistant whose goal is to help the user.

An agent has three parts:

- Instructions, the role and behavior we want. We pass this as the developer message. The better the instructions, the better the agent helps.
- Tools, the functions the agent can call to carry out the task. For us that's only search().
- Memory, the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried.

### A function-call helper
We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. We only have one tool for now, so we dispatch on the function name directly.

In [1]:
import json

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

We define `search()` again here so this notebook can run on its own. It searches only the LLM Zoomcamp FAQ and returns the five highest-ranked results.

In [2]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

The helper returns the exact structure the Responses API expects. When we add more tools later, we'll extend this with more if branches (or switch to a registry).

### Processing one response

Let's process one model response by hand before introducing the loop. We append every output item to the conversation. When an item is a function call, we execute it and append the corresponding function-call output as well. The model has not seen that tool result yet; it will receive it in the next API request.

Load environment variables and create the OpenAI client. By default, the client reads `OPENAI_API_KEY` from the environment.

In [3]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Load the FAQ documents and build the local search index used by `search()`.

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

Describe `search()` to the model as a tool. The JSON schema says that every call must contain one string argument named `query` and no additional arguments.

In [5]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

### Encouraging multiple searches

The model often answers after the first search, even when more searches would help. It reasons that it already knows enough, so why bother. We push it to explore more by providing the instructions in a developer prompt...

In [6]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

The instructions are how we steer the agent. It can still decide to skip ahead sometimes, so don't expect it to follow them every single run.

In [7]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

Response(id='resp_04312d611e865376006a6b29890cf881a1bf718c055e6ee146', created_at=1785407881.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5.4-mini-2026-03-17', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment late registration"}', call_id='call_wUTDd8VIiV3hMVYMhFcmekDi', name='search', type='function_call', id='fc_04312d611e865376006a6b298a117881a1be9fa185f86a3015', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"query":"course FAQ enrollment join discovered the course can I still join"}', call_id='call_rJiA7U2T8R8V9IZlP2km3J3S', name='search', type='function_call', id='fc_04312d611e865376006a6b298a118c81a18b788ded5c841f78', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'S

The `has_function_calls` flag tells us whether the agent needs another API request. If this response contains a function call, `messages` now contains both the call and its result. Sending the updated list back lets the model inspect the result and decide whether to search again or answer.

### The full agent loop

We now wrap the same steps in a `while` loop. Each iteration is one round-trip to the model. The loop continues while the response contains function calls and stops when the model produces a response with no more calls.

This is the core agent loop. The model reasons about the next action. Your code performs it, and the model sees the result on the next turn. The loop stops when the model returns a final answer with no more tool calls.

We do not decide in advance how many searches to perform. The model chooses when to search and whether another search is useful; our code is responsible for executing each request and preserving the resulting history.

This first version deliberately uses the simplest exit condition: no function calls means the loop is finished. Because an unrestricted loop could keep making API requests, the reusable version below adds a maximum iteration count and raises an error if the model does not produce a final answer within that limit.

In [8]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered the course can I join enrollment registration late join"}
iteration #2...
ASSISTANT:
Yes — you can still join the course. You can start learning anytime, even if you just discovered it.

One important note: if you want a certificate, you need to submit your project while the course is still accepting submissions.

If you’d like, I can also help you figure out how to start, where to find the materials, or what the weekly workflow looks like. Is there anything else you want to explore?


### Wrapping it in a function

Before making the loop reusable, `ensure_response_completed()` checks the status of every API response. A status of `completed` means that one API request finished successfully; it does not mean the whole agent task is finished. A completed response may still contain function calls. `incomplete`, `failed`, or unexpected statuses stop the loop with a clear error instead of allowing partial output into the conversation.

`agent_loop()` takes the instructions and question as parameters and returns the final text from `response.output_text`. It allows at most five iterations by default. If the model returns neither a function call nor text, or if it reaches the iteration limit without answering, the function raises an error.

In [9]:
def ensure_response_completed(response):
    if response.status == "completed":
        return

    if response.status == "incomplete":
        details = response.incomplete_details or "No details provided"
        raise RuntimeError(f"Response was incomplete: {details}")

    if response.status == "failed":
        details = response.error or "No error details provided"
        raise RuntimeError(f"Response failed: {details}")

    raise RuntimeError(f"Unexpected response status: {response.status}")

In [10]:
def agent_loop(
    instructions,
    question,
    model="gpt-5.4-mini",
    max_iterations=5,
) -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    for iteration in range(1, max_iterations + 1):
        print(f"iteration #{iteration}...")

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]    # search_tool is declared in the global scope
        )

        ensure_response_completed(response)
        messages.extend(response.output)

        has_function_calls = False

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                messages.append(make_call(item))
                has_function_calls = True

        if not has_function_calls:
            final_answer = response.output_text.strip()

            if not final_answer:
                raise RuntimeError(
                    "The response completed without a function call "
                    "or a text answer."
                )

            return final_answer

    raise RuntimeError(
        f"Agent did not produce a final answer after "
        f"{max_iterations} iterations."
    )

Trying it with a question that has a typo...

In [11]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally run install local FAQ"}
iteration #2...
function_call: search {"query":"Ollama run llama3 local server localhost 11434 python client course FAQ"}
iteration #3...


'Yes — you can run Ollama locally.\n\nQuick setup:\n1. Install Ollama from https://ollama.com/download\n   - macOS: install the `.pkg`\n   - Windows: install the `.msi`\n   - Linux:\n   ```bash\n   curl -fsSL https://ollama.com/install.sh | sh\n   ```\n\n2. Start a model locally:\n```bash\nollama run llama3\n```\n\nThis will download the model and open a local chat interface.\n\n3. Verify the local server is running:\n```bash\ncurl http://localhost:11434\n```\n\n4. If you want to use it from Python:\n```bash\npip install ollama\n```\n\nExample:\n```python\nimport ollama\n\nresponse = ollama.chat(\n    model=\'llama3\',\n    messages=[{"role": "user", "content": "Hello!"}]\n)\n\nprint(response[\'message\'][\'content\'])\n```\n\nIf you get a connection error, restart the Ollama server:\n```bash\nollama serve\n```\nor, in a notebook:\n```bash\n!nohup ollama serve > nohup.out 2>&1 &\n```\n\nIf you want, I can also show you how to use Ollama with a RAG app or inside a notebook.'

### Restricting off-topic questions

Right now the agent can answer from its general knowledge, even when the question has nothing to do with the course. Ask it about chess and it will still try to help.



In [12]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit basics"}
function_call: search {"query":"queen gambit course FAQ chess opening"}
iteration #2...


'The Queen’s Gambit is a chess opening that starts with:\n\n1. d4 d5  \n2. c4\n\nWhite offers the c-pawn to try to control the center and tempt Black into taking it. It’s one of the most famous and solid chess openings.\n\nThere are two main versions:\n- Queen’s Gambit Accepted: Black takes the c-pawn\n- Queen’s Gambit Declined: Black does not take it\n\nIf you want, I can also explain the ideas behind it in simple terms or show a few common lines. Any other areas you want to explore?'

We want a course assistant, not a general chatbot, so we tighten the instructions and require answers to be grounded in the FAQ. For another application, answering from general knowledge might be acceptable; here the restriction illustrates how instructions define an agent's scope.

Notice that a ranked search can return weak or unrelated matches rather than an empty list. The agent should therefore judge whether the retrieved FAQ entries actually support an answer instead of treating the presence of any result as proof that the question is on-topic.

In [13]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [14]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen's gambit"}
iteration #2...
function_call: search {"query":"queen gambit queen's gambit chess course FAQ"}
iteration #3...


'I couldn’t find any FAQ entry for “queen gambit” in the course materials, so it looks like this isn’t a course-related question.\n\nIf you meant something else related to the course, feel free to ask, and I can help with that. Is there anything else you’d like to explore?'

This is a lightweight, instruction-based guardrail: we tell the agent what is in scope and require it to ground answers in the FAQ. A dedicated input guardrail would classify the question before running the agent and could block off-topic requests outright.

This handwritten loop exposes the core pattern that agent frameworks build on: ask the model what to do, execute requested tools, append their results to memory, and repeat until the model answers or a safety limit is reached.